# Ecommerce Customer Churn Prediction: 
# Task 1: Basic Machine Learning Pipeline



## 1. Importing Required Libraries
The first step is to import Python libraries for data handling, preprocessing, model building, and evaluation.

In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Load and Inspect the Dataset

The dataset used is an ecommerce customer churn dataset. Each row represents one customer, and the goal is to predict whether the customer churned.

In [18]:
# Load the dataset
df = pd.read_csv("ecommerce_customer_churn_dataset.csv")

# Display the first five rows
df.head()

,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


In [19]:
# Check the number of rows and columns
df.shape

(325, 14)

In [20]:
# View column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               325 non-null    int64  
 1   Age                       320 non-null    float64
 2   Gender                    320 non-null    object 
 3   Region                    320 non-null    object 
 4   Membership_Type           320 non-null    object 
 5   Monthly_Spending          320 non-null    float64
 6   Number_of_Orders          325 non-null    int64  
 7   Days_Since_Last_Purchase  325 non-null    int64  
 8   Customer_Support_Calls    325 non-null    int64  
 9   Average_Rating            320 non-null    float64
 10  Used_Coupon               325 non-null    object 
 11  Newsletter_Subscribed     325 non-null    object 
 12  Device_Type               325 non-null    object 
 13  Churned                   325 non-null    object 
dtypes: float64

In [21]:
# Summary statistics for numerical columns
df.describe()

,Customer_ID,Age,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating
count,325.000000,320.00000,320.000000,325.000000,325.000000,325.000000,320.000000
mean,159.950769,43.80000,603.018750,11.956923,177.803077,4.415385,2.943750
std,92.311356,14.91409,345.057389,7.205407,107.633715,2.945452,1.410871
min,1.000000,18.00000,27.000000,0.000000,1.000000,0.000000,1.000000
25%,80.000000,31.00000,306.250000,6.000000,81.000000,2.000000,2.000000
50%,160.000000,43.00000,599.500000,13.000000,181.000000,4.000000,3.000000
75%,239.000000,56.25000,892.000000,18.000000,273.000000,7.000000,4.000000
max,320.000000,69.00000,1193.000000,24.000000,363.000000,9.000000,5.000000


In [22]:
# Check missing values in each column
df.isnull().sum()

Customer_ID                 0
Age                         5
Gender                      5
Region                      5
Membership_Type             5
Monthly_Spending            5
Number_of_Orders            0
Days_Since_Last_Purchase    0
Customer_Support_Calls      0
Average_Rating              5
Used_Coupon                 0
Newsletter_Subscribed       0
Device_Type                 0
Churned                     0
dtype: int64

## 3. Understand the Business Problem

The business problem is to predict **customer churn** for an ecommerce company. Churn means that a customer stops buying from the company or stops using the service.

This is useful because the company can identify customers who are likely to leave and take action early, such as sending offers, improving customer support, or creating retention campaigns.

## 4. Identify Features and Target Variable

The **target variable** is `Churned`, because this is what the model is trying to predict.

The **features** are the customer-related variables that may help explain churn, such as age, spending, number of orders, membership type, support calls, rating, coupon use, newsletter subscription, and device type.

`Customer_ID` is removed because it is only an identifier and does not provide useful predictive information.


In [23]:
# Define features X and target y
X = df.drop(columns=["Customer_ID", "Churned"])
y = df["Churned"]

print("Feature columns:")
print(X.columns.tolist())

print("Target variable:")
print(y.name)

Feature columns:
['Age', 'Gender', 'Region', 'Membership_Type', 'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase', 'Customer_Support_Calls', 'Average_Rating', 'Used_Coupon', 'Newsletter_Subscribed', 'Device_Type']
Target variable:
Churned


In [24]:
# Convert target variable into binary values
# Yes = 1 means the customer churned
# No = 0 means the customer did not churn
y = y.map({"No": 0, "Yes": 1})
y.value_counts()

Churned
0    238
1     87
Name: count, dtype: int64

## 5. Clean the Data

The dataset may contain missing values. Instead of manually filling them before the train-test split, we use a preprocessing pipeline. This is a better practice because preprocessing is fitted only on the training data, which helps avoid data leakage.

In [25]:
# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['Age', 'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase', 'Customer_Support_Calls', 'Average_Rating']
Categorical features: ['Gender', 'Region', 'Membership_Type', 'Used_Coupon', 'Newsletter_Subscribed', 'Device_Type']


## 6. Split the Data into Training and Testing Sets

The data is split into training and testing sets. The training set is used to train the model, while the testing set is used to evaluate how well the model performs on unseen data.

A stratified split is used so that the churn and non-churn classes are represented fairly in both

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training feature shape: (260, 12)
Testing feature shape: (65, 12)
Training target shape: (260,)
Testing target shape: (65,)


## 7. Apply Preprocessing

The preprocessing step handles both numerical and categorical columns:

- Numerical columns: missing values are filled using the median, then values are scaled.
- Categorical columns: missing values are filled using the most frequent value, then categories are converted using one-hot encoding.

In [27]:
# Preprocessing for numerical columns
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

# Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)])

## 8. Train a Baseline Logistic Regression Model

Logistic Regression is used as a simple baseline model because the target variable is binary: churned or not churned. The preprocessing and model are combined into one pipeline.

In [28]:
# Create the full machine learning pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Monthly_Spending',
                                                   'Number_of_Orders',
                                                   'Days_Since_Last_Purchase',
                                                   'Customer_Support_Calls',
                                                   'Average_Rating']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Region',
                                                   'Membership_Type',
                                                   'Used_Coupon',
                                                   'Newsletter_Subscribed',
                                                   'Device_Type'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

## 9. Make Predictions

In [29]:
# Predict on the test set
y_pred = model.predict(X_test)
y_pred[:10]

array([0, 1, 1, 0, 0, 0, 0, 1, 0, 0])

## 10. Evaluate the Model

The model is evaluated using accuracy, precision, recall, F1-score, confusion matrix, and classification report.

In [30]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

Accuracy: 0.8154
Precision: 0.6471
Recall: 0.6471
F1-score: 0.6471


In [31]:
# Confusion matrix
confusion_matrix(y_test, y_pred)

array([[42,  6],
       [ 6, 11]])

In [32]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=["Not Churned", "Churned"]))

              precision    recall  f1-score   support

 Not Churned       0.88      0.88      0.88        48
     Churned       0.65      0.65      0.65        17

    accuracy                           0.82        65
   macro avg       0.76      0.76      0.76        65
weighted avg       0.82      0.82      0.82        65



# Task 2: Explanation and Submission Summary

## What the Model is Trying to Predict
The objective of this machine learning model is to predict customer churn in an ecommerce business. Customer churn refers to customers who stop purchasing products or using the platform’s services. In this dataset, churn is represented by the target variable `Churned`, where:

- `Yes` → Customer churned
- `No` → Customer remained active

Predicting churn is important because it helps businesses identify customers who are at risk of leaving and allows companies to take preventive actions such as targeted promotions, loyalty rewards, or improved customer support.



## Dataset Used
The dataset used for this project is the **Ecommerce Customer Churn Dataset**. The dataset contains customer demographic information, purchasing behavior, engagement metrics, and service-related details. These variables help analyze customer patterns and build a predictive model for churn classification.



## Features and Target Variable

### Target Variable
The target variable selected for prediction was:

```python
Churned
```

### Features Selected
The input features included multiple customer-related variables such as:

- Age
- Gender
- Region
- Membership Type
- Monthly Spending
- Number of Orders
- Days Since Last Purchase
- Customer Support Calls
- Average Rating
- Coupon Usage
- Newsletter Subscription
- Device Type

The column `Customer_ID` was removed because it is only a unique identifier and does not provide meaningful information for prediction.



## Data Preprocessing and Pipeline
Several preprocessing steps were applied before training the model:

- Missing numerical values were handled using median imputation
- Missing categorical values were handled using most frequent value imputation
- Numerical variables were standardized using `StandardScaler`
- Categorical variables were encoded using `OneHotEncoder`
- The dataset was split into training and testing sets using an 80-20 ratio

A complete machine learning pipeline was created using Scikit-learn’s `Pipeline` and `ColumnTransformer` to ensure preprocessing and modeling steps were applied correctly and consistently.



## Model Used
A **Logistic Regression** model was selected as the baseline classification model because the target variable is binary (churned or not churned). Logistic Regression is simple, interpretable, and commonly used as an initial benchmark model in machine learning projects.



## Results Obtained
The Logistic Regression model produced the following evaluation results on the testing dataset:

- **Accuracy:** 0.8154
- **Precision:** 0.6471
- **Recall:** 0.6471
- **F1-score:** 0.6471

These results indicate that the model was able to predict customer churn with reasonably good performance using the available customer information.



## Limitation of the Model or Dataset
One limitation of this project is that Logistic Regression is a relatively simple linear model and may not capture complex non-linear customer behavior patterns. More advanced models such as Random Forest, XGBoost, or Neural Networks may potentially achieve better predictive performance.

Additionally, the dataset may not contain all factors influencing customer churn. External factors such as competitor pricing, customer satisfaction history, economic conditions, and marketing campaign effectiveness were not included in the dataset.



## Final Conclusion
This project successfully developed a complete end-to-end machine learning pipeline for customer churn prediction. The notebook covered all major stages of a machine learning workflow, including:

- Loading and inspecting the dataset
- Understanding the business problem
- Defining features and target variables
- Data cleaning and preprocessing
- Handling missing values and categorical variables
- Splitting the dataset into training and testing sets
- Building and training a Logistic Regression baseline model
- Evaluating model performance

Overall, the project demonstrates a structured and professional implementation of a machine learning classification pipeline using Python and Scikit-learn.
